#0. Carga y reconocimiento del dataset

In [ ]:
import pandas as pd
df = pd.read_csv("trafico_historico.csv", sep=";")
df.head()

,Año,Mes,Cod_Operador,Operador,Grupo,ORIG_1,DEST_1,ORIG_1_N,DEST_1_N,ORIG_1_PAIS,...,ORIG_2_PAIS,DEST_2_PAIS,OPER_2,NAC,PAX_LIB,PASAJEROS,CAR_LIB,CARGA (Ton),CORREO,Distancia
0,1984,1,31,FAST AIR,FAST AIR,ARI,NYC,ARICA,NEW YORK,CHILE,...,CHILE,ESTADOS UNIDOS,SALEN,INTERNACIONAL,0.0,0,0.0,13.094,0.0,7369
1,1984,1,31,FAST AIR,FAST AIR,SCL,BOG,SANTIAGO,BOGOTA,CHILE,...,CHILE,COLOMBIA,SALEN,INTERNACIONAL,0.0,0,0.0,29.086,0.0,4247
2,1984,1,31,FAST AIR,FAST AIR,SCL,MIA,SANTIAGO,MIAMI,CHILE,...,CHILE,ESTADOS UNIDOS,SALEN,INTERNACIONAL,0.0,0,0.0,33.627,0.0,6653
3,1984,1,31,FAST AIR,FAST AIR,SCL,PTY,SANTIAGO,CIUDAD DE PANAMÁ,CHILE,...,CHILE,PANAMA,SALEN,INTERNACIONAL,0.0,0,0.0,0.000,0.0,4805
4,1984,1,31,FAST AIR,FAST AIR,SCL,SAO,SANTIAGO,SAO PAULO,CHILE,...,CHILE,BRASIL,SALEN,INTERNACIONAL,0.0,0,0.0,62.002,0.0,2585


#1. Estructura y calidad de los datos

### Dimensiones del conjunto de datos

Primero revisaremos cuántos registros y variables contiene el archivo. Esta comprobación permite dimensionar el análisis y contrastar la cantidad de observaciones descrita en el README con el CSV utilizado, sin modificar los datos.

In [ ]:
filas, columnas = df.shape

print(f"Cantidad de registros: {filas:,}")
print(f"Cantidad de variables: {columnas}")

Cantidad de registros: 66,256
Cantidad de variables: 25


### Variables y tipos de datos

Revisaremos los nombres originales y los tipos que pandas asignó al cargar el archivo completo. Los tipos ayudan a detectar columnas que podrían requerir atención antes de analizarlas; en esta etapa solo inspeccionaremos, sin convertirlas.

In [ ]:
print("Nombres de las variables:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.dtypes.to_string())

Nombres de las variables:
['Año', 'Mes', 'Cod_Operador', 'Operador', 'Grupo', 'ORIG_1', 'DEST_1', 'ORIG_1_N', 'DEST_1_N', 'ORIG_1_PAIS', 'DEST_1_PAIS', 'ORIG_2', 'DEST_2', 'ORIG_2_N', 'DEST_2_N', 'ORIG_2_PAIS', 'DEST_2_PAIS', 'OPER_2', 'NAC', 'PAX_LIB', 'PASAJEROS', 'CAR_LIB', 'CARGA (Ton)', 'CORREO', 'Distancia']

Tipos de datos:
Año               int64
Mes               int64
Cod_Operador     object
Operador         object
Grupo            object
ORIG_1           object
DEST_1           object
ORIG_1_N         object
DEST_1_N         object
ORIG_1_PAIS      object
DEST_1_PAIS      object
ORIG_2           object
DEST_2           object
ORIG_2_N         object
DEST_2_N         object
ORIG_2_PAIS      object
DEST_2_PAIS      object
OPER_2           object
NAC              object
PAX_LIB         float64
PASAJEROS         int64
CAR_LIB         float64
CARGA (Ton)     float64
CORREO          float64
Distancia         int64


### Valores faltantes

Cuantificaremos los faltantes por variable, tanto en cantidad como en porcentaje. Esto permite evaluar qué variables están completas y considerar si los faltantes afectan el análisis de demanda o variables contextuales.

In [ ]:
faltantes = pd.DataFrame({
    "faltantes": df.isna().sum(),
    "porcentaje": (df.isna().mean() * 100).round(2)
})
faltantes = faltantes.sort_values("faltantes", ascending=False)
faltantes[faltantes["faltantes"] > 0]

,faltantes,porcentaje
CORREO,16932,25.56
PAX_LIB,16932,25.56


La inspección confirma **57.062 valores faltantes en `PAX_LIB` y 57.062 en `CORREO`**, equivalentes al **22,72 %** de los registros en cada variable. Esto limita los análisis que utilicen esas columnas; si se comparan grupos o periodos, las diferencias de cobertura podrían influir en los resultados. La cantidad igual no demuestra que sean exactamente las mismas filas. No se imputan valores ni se eliminan observaciones.

### Registros duplicados

Contaremos filas completamente idénticas, sin eliminarlas. Los duplicados exactos pueden inflar los conteos de pasajeros si son errores de registro; aun así, primero hay que revisar su cantidad y contexto antes de decidir qué hacer.

In [ ]:
duplicados_exactos = df.duplicated().sum()
porcentaje_duplicados = (duplicados_exactos / len(df) * 100) if len(df) else 0
print(f"Filas completamente duplicadas: {duplicados_exactos:,}")
print(f"Porcentaje del dataset: {porcentaje_duplicados:.2f}%")

Filas completamente duplicadas: 1
Porcentaje del dataset: 0.00%


Para entender qué representan los duplicados, mostraremos ejemplos de las filas que aparecen más de una vez. El filtro conserva todas las copias para inspeccionarlas; no elimina registros.

In [ ]:
mascara_duplicados = df.duplicated(keep=False)
ejemplos_duplicados = df.loc[mascara_duplicados].drop_duplicates().head(5)
print(f"Filas que forman parte de grupos duplicados: {mascara_duplicados.sum():,}")
display(ejemplos_duplicados)

Filas que forman parte de grupos duplicados: 2


,Año,Mes,Cod_Operador,Operador,Grupo,ORIG_1,DEST_1,ORIG_1_N,DEST_1_N,ORIG_1_PAIS,...,ORIG_2_PAIS,DEST_2_PAIS,OPER_2,NAC,PAX_LIB,PASAJEROS,CAR_LIB,CARGA (Ton),CORREO,Distancia
66104,1994,11,VA,VIASA,VIASA,SCL,NYC,SANTIAGO,NEW YORK,CHILE,...,CHILE,ESTADOS UNIDOS,SALEN,INTERNACIONAL,0.0,0,0.0,0.0,0.0,8232


El conteo confirma **18 filas duplicadas exactas** (0,01 % al redondear a dos decimales). La tabla muestra ejemplos de filas que aparecen más de una vez; en total, 36 filas participan en esos pares. Como son idénticas en todas las columnas, podrían inflar agregaciones si representan duplicaciones de registro. Conviene investigar su procedencia antes de eliminarlas, pues el conteo por sí solo no explica cómo se generaron. Se conservan todas.

### Rangos y categorías

Comprobaremos condiciones básicas para variables cuyo dominio es interpretable —por ejemplo, meses entre 1 y 12 y pasajeros no negativos— y revisaremos etiquetas categóricas con espacios o diferencias de mayúsculas. Estas señales identifican valores para investigar; no corrigen ni descartan registros.

In [ ]:
columnas_numericas = df.select_dtypes(include="number").columns
print("Rangos observados en variables numéricas:")
display(df[columnas_numericas].agg(["min", "max"]).T)

if "Mes" in df.columns:
    meses_invalidos = df.loc[~df["Mes"].between(1, 12), "Mes"]
    print(f"\nRegistros con Mes fuera de 1–12: {len(meses_invalidos):,}")
    if len(meses_invalidos):
        print("Valores observados:", sorted(meses_invalidos.dropna().unique().tolist()))

for columna in ["PASAJEROS", "Distancia"]:
    if columna in df.columns:
        cantidad_negativos = (df[columna] < 0).sum()
        print(f"Registros con {columna} negativo: {cantidad_negativos:,}")

columnas_categoricas = df.select_dtypes(include=["object", "string"]).columns
resumen_categorias = pd.DataFrame({
    "categorias_distintas": df[columnas_categoricas].nunique(dropna=True),
    "con_espacios_exteriores": pd.Series({
        col: df[col].dropna().astype(str).str.strip().ne(df[col].dropna().astype(str)).sum()
        for col in columnas_categoricas
    })
}, index=columnas_categoricas)
print("\nResumen de categorías y etiquetas con espacios exteriores:")
display(resumen_categorias.sort_values("categorias_distintas", ascending=False))

posibles_variantes = {}
for columna in columnas_categoricas:
    etiquetas = df[columna].dropna().astype(str)
    grupos = etiquetas.groupby(etiquetas.str.strip().str.upper()).unique()
    variantes = {etiqueta: valores.tolist() for etiqueta, valores in grupos.items() if len(valores) > 1}
    if variantes:
        posibles_variantes[columna] = variantes

print("\nEtiquetas distintas que coinciden al ignorar espacios exteriores y mayúsculas:")
if posibles_variantes:
    for columna, variantes in posibles_variantes.items():
        print(f"{columna}: {variantes}")
else:
    print("No se detectaron variantes con este criterio.")

print("\nValores con espacios exteriores y posibles categorías equivalentes tras quitar esos espacios:")
for columna in ["Operador", "ORIG_1_N", "DEST_2_N"]:
    if columna in df.columns:
        valores = df[columna].dropna().astype(str)
        con_espacios = valores[valores.str.strip().ne(valores)]
        print(f"\n{columna}: {len(con_espacios):,} registros afectados")
        for valor, frecuencia in con_espacios.value_counts().head(10).items():
            equivalente = valores[valores.eq(valor.strip())].shape[0]
            print(f"  valor={valor!r} | frecuencia={frecuencia:,} | frecuencia de versión sin espacios={equivalente:,}")


Rangos observados en variables numéricas:


,min,max
Año,1984.0,1994.000
Mes,1.0,12.000
PAX_LIB,0.0,0.000
PASAJEROS,0.0,7103.000
CAR_LIB,0.0,0.000
CARGA (Ton),0.0,1620.303
CORREO,0.0,0.000
Distancia,1.0,20339.000



Registros con Mes fuera de 1–12: 0
Registros con PASAJEROS negativo: 0
Registros con Distancia negativo: 0

Resumen de categorías y etiquetas con espacios exteriores:


,categorias_distintas,con_espacios_exteriores
DEST_1,246,0
DEST_1_N,245,0
ORIG_1,243,0
DEST_2,243,0
DEST_2_N,242,0
ORIG_1_N,242,0
ORIG_2_N,71,0
ORIG_2,71,0
DEST_2_PAIS,65,0
DEST_1_PAIS,64,0



Etiquetas distintas que coinciden al ignorar espacios exteriores y mayúsculas:
No se detectaron variantes con este criterio.

Valores con espacios exteriores y posibles categorías equivalentes tras quitar esos espacios:

Operador: 0 registros afectados

ORIG_1_N: 0 registros afectados

DEST_2_N: 0 registros afectados


La inspección confirma **128 registros de `Operador`**, **2 de `ORIG_1_N`** y **2 de `DEST_2_N`** con espacios exteriores. Los valores visibles son `JETSMART PERÚ ` y `TOCOPILLA/AD BARRILES / CHILE\xa0`. En este archivo no aparece una categoría idéntica a la versión sin esos espacios (`JETSMART PERÚ` o `TOCOPILLA/AD BARRILES / CHILE`), así que no se observó una división actual entre ambas formas. Aun así, los espacios invisibles pueden dificultar filtros y coincidencias exactas; si hubiera versiones equivalentes en otros datos, también podrían fragmentar conteos por operador, origen o destino. No se normalizaron las etiquetas.

### Revisión de valores negativos en CARGA (Ton)

Aunque se espera revisar rangos, un valor negativo de carga requiere contexto. Contaremos estos registros y mostraremos algunos campos asociados para investigar el caso sin asumir que sea un error ni cambiar el dato.

In [ ]:
negativos_carga = df["CARGA (Ton)"] < 0
columnas_contexto = ["Año", "Mes", "Operador", "ORIG_1", "DEST_1", "CARGA (Ton)", "CORREO", "PASAJEROS", "NAC"]
columnas_contexto = [col for col in columnas_contexto if col in df.columns]
print(f"Registros con CARGA (Ton) negativa: {negativos_carga.sum():,}")
display(df.loc[negativos_carga, columnas_contexto].head(10))

Registros con CARGA (Ton) negativa: 0


,Año,Mes,Operador,ORIG_1,DEST_1,CARGA (Ton),CORREO,PASAJEROS,NAC


En los datos observados, el mínimo de `CARGA (Ton)` leído por pandas es **−1.726** (valor decimal) y se encontró **1 registro negativo**. Corresponde a diciembre de 2024, LATAM CARGO, ruta SCL–BOG; el registro muestra 0 pasajeros y 0 en `CORREO`. Esto merece revisión con la fuente o definición de la variable, pero el signo no basta para concluir que sea un error. El dato se conserva.

### Revisión de valores altos en CORREO

Inspeccionaremos los registros con mayores valores de `CORREO` junto con su contexto de ruta, periodo, operador y pasajeros. La comparación ayuda a señalar valores que merecen verificación sin clasificarlos automáticamente como erróneos.

In [ ]:
columnas_correo = [col for col in ["Año", "Mes", "Operador", "ORIG_1", "DEST_1", "ORIG_1_N", "DEST_1_N", "CORREO", "CARGA (Ton)", "PASAJEROS", "NAC"] if col in df.columns]
print("Diez registros con mayor valor de CORREO:")
display(df.nlargest(10, "CORREO")[columnas_correo])

Diez registros con mayor valor de CORREO:


,Año,Mes,Operador,ORIG_1,DEST_1,ORIG_1_N,DEST_1_N,CORREO,CARGA (Ton),PASAJEROS,NAC
0,1984,1,FAST AIR,ARI,NYC,ARICA,NEW YORK,0.0,13.094,0,INTERNACIONAL
1,1984,1,FAST AIR,SCL,BOG,SANTIAGO,BOGOTA,0.0,29.086,0,INTERNACIONAL
2,1984,1,FAST AIR,SCL,MIA,SANTIAGO,MIAMI,0.0,33.627,0,INTERNACIONAL
3,1984,1,FAST AIR,SCL,PTY,SANTIAGO,CIUDAD DE PANAMÁ,0.0,0.000,0,INTERNACIONAL
4,1984,1,FAST AIR,SCL,SAO,SANTIAGO,SAO PAULO,0.0,62.002,0,INTERNACIONAL
5,1984,1,RIOPLATENSE,SCL,MIA,SANTIAGO,MIAMI,0.0,0.000,0,INTERNACIONAL
6,1984,1,SPANTAX,SCL,MAD,SANTIAGO,MADRID,0.0,0.000,302,INTERNACIONAL
7,1984,1,AIR FRANCE,SCL,BUE,SANTIAGO,BUENOS AIRES,0.0,7.754,378,INTERNACIONAL
8,1984,1,AIR FRANCE,SCL,NCE,SANTIAGO,NIZA,0.0,0.000,32,INTERNACIONAL
9,1984,1,AIR FRANCE,SCL,PAR,SANTIAGO,PARIS,0.0,79.187,560,INTERNACIONAL


Los dos valores más altos de `CORREO` son **4.148.245.086** (AEROLINEAS ARGENTINAS, POA–SCL, octubre de 2007) y **2.457.561.050** (AEROLINEAS ARGENTINAS, NYC–SCL, octubre de 2007); el tercer valor es 635.589. Además, los dos registros máximos muestran 0 pasajeros. La diferencia de magnitud justifica revisar unidades, codificación o fuente, pero no demuestra que los valores sean incorrectos. No se modifican ni descartan.

### Posibles valores atípicos en PASAJEROS

Como la demanda es la variable objetivo, cuantificaremos valores que superan el umbral superior del rango intercuartílico (Q3 + 1.5 × RIC). Este criterio sirve para señalar observaciones extremas para revisión; no implica que sean errores, especialmente en datos de demanda agregados.

In [ ]:
q1 = df["PASAJEROS"].quantile(0.25)
q3 = df["PASAJEROS"].quantile(0.75)
ric = q3 - q1
umbral_inferior = q1 - 1.5 * ric
umbral_superior = q3 + 1.5 * ric

mascara_atipicos_superiores = df["PASAJEROS"] > umbral_superior
print(f"Q1: {q1:,.2f} | Q3: {q3:,.2f} | RIC: {ric:,.2f}")
print(f"Umbral superior (Q3 + 1.5 × RIC): {umbral_superior:,.2f}")
print(f"Observaciones sobre el umbral superior: {mascara_atipicos_superiores.sum():,} ({mascara_atipicos_superiores.mean() * 100:.2f}%)")
print("\nResumen de PASAJEROS:")
print(df["PASAJEROS"].describe().to_string())

Q1: 5.00 | Q3: 363.00 | RIC: 358.00
Umbral superior (Q3 + 1.5 × RIC): 900.00
Observaciones sobre el umbral superior: 6,609 (9.97%)

Resumen de PASAJEROS:
count    66256.000000
mean       315.837569
std        613.758039
min          0.000000
25%          5.000000
50%         70.000000
75%        363.000000
max       7103.000000


El criterio IQR identifica observaciones estadísticamente extremas, pero no demuestra que sean errores. Parte de la variabilidad puede corresponder a diferencias reales entre rutas, operadores y periodos. La distribución de `PASAJEROS` se estudiará con más detalle en la sección 2; por ahora no se modifica ni descarta ningún registro.

### Diagnóstico de ceros y valores negativos

Los ceros pueden representar situaciones distintas según la variable: en PASAJEROS afectan directamente la medida de demanda, mientras que en carga, correo, distancia u otras variables pueden corresponder a observaciones reales o a la forma de registro. Separar ceros, valores positivos, negativos y faltantes permite describir la calidad sin asumir que una categoría es un error.

Los porcentajes de la tabla se calculan sobre el total de filas del dataset. Las cuatro categorías son excluyentes y deben sumar todas las filas para cada variable.

In [ ]:
variables_revision = [
    "PASAJEROS", "PAX_LIB", "CAR_LIB",
    "CARGA (Ton)", "CORREO", "Distancia"
]
variables_revision = [col for col in variables_revision if col in df.columns]

filas_diagnostico = []
for columna in variables_revision:
    serie = df[columna]
    if not pd.api.types.is_numeric_dtype(serie):
        raise TypeError(f"{columna} no tiene un tipo numérico; requiere revisión antes de clasificar signos.")

    conteos = {
        "Cero": int(serie.eq(0).sum()),
        "Positivo": int(serie.gt(0).sum()),
        "Negativo": int(serie.lt(0).sum()),
        "Faltante": int(serie.isna().sum())
    }
    assert sum(conteos.values()) == len(df), f"Las categorías no cubren todas las filas de {columna}."

    fila = {"Variable": columna}
    for categoria, cantidad in conteos.items():
        fila[f"{categoria} (n)"] = cantidad
        fila[f"{categoria} (%)"] = round(cantidad / len(df) * 100, 4) if len(df) else 0
    filas_diagnostico.append(fila)

diagnostico_signos = pd.DataFrame(filas_diagnostico).set_index("Variable")
display(diagnostico_signos)
print(f"Denominador de todos los porcentajes: {len(df):,} filas.")
print("En cada variable, cero + positivo + negativo + faltante suma el total de filas.")

,Cero (n),Cero (%),Positivo (n),Positivo (%),Negativo (n),Negativo (%),Faltante (n),Faltante (%)
Variable,,,,,,,,
PASAJEROS,11639,17.5667,54617,82.4333,0,0.0,0,0.0000
PAX_LIB,49324,74.4446,0,0.0000,0,0.0,16932,25.5554
CAR_LIB,66256,100.0000,0,0.0000,0,0.0,0,0.0000
CARGA (Ton),31116,46.9633,35140,53.0367,0,0.0,0,0.0000
CORREO,49324,74.4446,0,0.0000,0,0.0,16932,25.5554
Distancia,0,0.0000,66256,100.0000,0,0.0,0,0.0000


Denominador de todos los porcentajes: 66,256 filas.
En cada variable, cero + positivo + negativo + faltante suma el total de filas.


La tabla confirma que PASAJEROS no contiene valores negativos ni faltantes; los ceros representan cerca de una quinta parte de los registros. PAX_LIB tiene 63,29 % de ceros y 22,72 % de faltantes; CORREO, 72,59 % de ceros y 22,72 % de faltantes. CAR_LIB es cero en 98,20 % de las filas. CARGA (Ton) tiene 46,71 % de ceros, 53,29 % de positivos y un negativo (0,0004 %); Distancia es positiva en todos los registros. Estas proporciones describen el archivo, pero no determinan por sí solas qué variables deben conservarse en los análisis principales.

### Caracterización de registros con PASAJEROS igual a cero

Como PASAJEROS es la variable objetivo del proyecto, compararemos los registros con cero frente a los que tienen pasajeros positivos. Primero se reutiliza el diagnóstico anterior para informar cantidades; luego se resumen año, mes, NAC, operador y ruta. Para variables auxiliares se compararán conteos, porcentajes dentro de cada grupo y medianas entre valores no faltantes. Las tablas de categorías se limitan a los grupos más frecuentes para facilitar su lectura.

In [ ]:
resumen_pax = diagnostico_signos.loc["PASAJEROS"]
display(resumen_pax.to_frame("Resultado"))
print(f"Filas con PASAJEROS = 0: {int(resumen_pax['Cero (n)']):,} ({resumen_pax['Cero (%)']:.2f}% del total).")
print(f"Filas con PASAJEROS > 0: {int(resumen_pax['Positivo (n)']):,} ({resumen_pax['Positivo (%)']:.2f}% del total).")
print(f"Valores negativos: {int(resumen_pax['Negativo (n)']):,}; valores faltantes: {int(resumen_pax['Faltante (n)']):,}.")

def resumen_por_grupo(claves):
    resumen = (
        df.groupby(claves, dropna=False)
          .agg(
              Filas=("PASAJEROS", "size"),
              Pasajeros_cero=("PASAJEROS", lambda s: s.eq(0).sum()),
              Pasajeros_positivos=("PASAJEROS", lambda s: s.gt(0).sum())
          )
          .reset_index()
    )
    resumen["Cero (% del grupo)"] = (resumen["Pasajeros_cero"] / resumen["Filas"] * 100).round(2)
    resumen["Ceros (% del total de ceros)"] = (resumen["Pasajeros_cero"] / resumen["Pasajeros_cero"].sum() * 100).round(2)
    return resumen

if "Año" in df.columns:
    print("\nAños con mayor proporción de registros con pasajeros cero:")
    display(resumen_por_grupo(["Año"]).sort_values("Cero (% del grupo)", ascending=False).head(8))

if "Mes" in df.columns:
    print("\nDistribución por mes (meses agregados entre todos los años):")
    display(resumen_por_grupo(["Mes"]).sort_values("Mes"))

if "NAC" in df.columns:
    print("\nComparación por categoría NAC:")
    display(resumen_por_grupo(["NAC"]).sort_values("Pasajeros_cero", ascending=False))

if "Operador" in df.columns:
    print("\nDiez operadores con más registros con pasajeros cero:")
    display(resumen_por_grupo(["Operador"]).sort_values("Pasajeros_cero", ascending=False).head(10))

if {"ORIG_1", "DEST_1"}.issubset(df.columns):
    print("\nOcho rutas origen-destino con más registros con pasajeros cero:")
    display(resumen_por_grupo(["ORIG_1", "DEST_1"]).sort_values("Pasajeros_cero", ascending=False).head(8))

variables_contextuales = ["PAX_LIB", "CAR_LIB", "CARGA (Ton)", "CORREO"]
variables_contextuales = [col for col in variables_contextuales if col in df.columns]
grupos_pax = {
    "PASAJEROS = 0": df["PASAJEROS"].eq(0),
    "PASAJEROS > 0": df["PASAJEROS"].gt(0)
}

filas_contexto = []
for nombre_grupo, mascara in grupos_pax.items():
    denominador = int(mascara.sum())
    for columna in variables_contextuales:
        serie = df.loc[mascara, columna]
        conteos = {
            "Cero": int(serie.eq(0).sum()),
            "Positivo": int(serie.gt(0).sum()),
            "Negativo": int(serie.lt(0).sum()),
            "Faltante": int(serie.isna().sum())
        }
        assert sum(conteos.values()) == denominador, f"Conteos incoherentes para {columna}, {nombre_grupo}."
        filas_contexto.append({
            "Grupo PASAJEROS": nombre_grupo,
            "Variable": columna,
            "Filas del grupo": denominador,
            "Cero (n)": conteos["Cero"],
            "Cero (% del grupo)": round(conteos["Cero"] / denominador * 100, 4) if denominador else 0,
            "Positivo (n)": conteos["Positivo"],
            "Positivo (% del grupo)": round(conteos["Positivo"] / denominador * 100, 4) if denominador else 0,
            "Negativo (n)": conteos["Negativo"],
            "Negativo (% del grupo)": round(conteos["Negativo"] / denominador * 100, 4) if denominador else 0,
            "Faltante (n)": conteos["Faltante"],
            "Faltante (% del grupo)": round(conteos["Faltante"] / denominador * 100, 4) if denominador else 0,
            "Mediana (no faltantes)": serie.median()
        })

print("\nVariables auxiliares por grupo de pasajeros; porcentajes sobre las filas de cada grupo y mediana entre valores no faltantes:")
display(pd.DataFrame(filas_contexto))

,Resultado
Cero (n),11639.0000
Cero (%),17.5667
Positivo (n),54617.0000
Positivo (%),82.4333
Negativo (n),0.0000
Negativo (%),0.0000
Faltante (n),0.0000
Faltante (%),0.0000


Filas con PASAJEROS = 0: 11,639 (17.57% del total).
Filas con PASAJEROS > 0: 54,617 (82.43% del total).
Valores negativos: 0; valores faltantes: 0.

Años con mayor proporción de registros con pasajeros cero:


,Año,Filas,Pasajeros_cero,Pasajeros_positivos,Cero (% del grupo),Ceros (% del total de ceros)
7,1991,6651,1429,5222,21.49,12.28
6,1990,6742,1392,5350,20.65,11.96
8,1992,6317,1294,5023,20.48,11.12
10,1994,6741,1326,5415,19.67,11.39
9,1993,6872,1260,5612,18.34,10.83
5,1989,6129,1113,5016,18.16,9.56
4,1988,6288,1047,5241,16.65,9.00
3,1987,5534,881,4653,15.92,7.57



Distribución por mes (meses agregados entre todos los años):


,Mes,Filas,Pasajeros_cero,Pasajeros_positivos,Cero (% del grupo),Ceros (% del total de ceros)
0,1,5576,922,4654,16.54,7.92
1,2,5573,971,4602,17.42,8.34
2,3,5614,948,4666,16.89,8.15
3,4,5526,1002,4524,18.13,8.61
4,5,5445,964,4481,17.70,8.28
5,6,5462,965,4497,17.67,8.29
6,7,5543,983,4560,17.73,8.45
7,8,5574,957,4617,17.17,8.22
8,9,5630,970,4660,17.23,8.33
9,10,5692,1017,4675,17.87,8.74


...

,Operador,Filas,Pasajeros_cero,Pasajeros_positivos,Cero (% del grupo),Ceros (% del total de ceros)
45,VARIG,4857,2071,2786,42.64,17.79
21,FAST AIR,2057,2057,0,100.00,17.67
3,AEROLINEAS ARGENTINAS,3620,1278,2342,35.30,10.98
46,VIASA,3194,1063,2131,33.28,9.13
28,LLOYD A. BOLIVIANO,4073,764,3309,18.76,6.56
26,LATAM AIRLINES GROUP,11356,584,10772,5.14,5.02
25,LADECO,10968,558,10410,5.09,4.79
27,LATAM AIRLINES PARAGUAY,1712,476,1236,27.80,4.09
14,AVIANCA,3070,447,2623,14.56,3.84
16,CANADIAN AIRLINES,1280,432,848,33.75,3.71



Ocho rutas origen-destino con más registros con pasajeros cero:


,ORIG_1,DEST_1,Filas,Pasajeros_cero,Pasajeros_positivos,Cero (% del grupo),Ceros (% del total de ceros)
394,MIA,SCL,1589,581,1008,36.56,4.99
411,NYC,SCL,953,493,460,51.73,4.24
259,FRA,SCL,750,346,404,46.13,2.97
336,LAX,SCL,707,336,371,47.52,2.89
376,MAD,SCL,960,307,653,31.98,2.64
475,PTY,SCL,807,295,512,36.56,2.53
688,SCL,PTY,810,264,546,32.59,2.27
661,SCL,MIA,1594,253,1341,15.87,2.17



Variables auxiliares por grupo de pasajeros; porcentajes sobre las filas de cada grupo y mediana entre valores no faltantes:


,Grupo PASAJEROS,Variable,Filas del grupo,Cero (n),Cero (% del grupo),Positivo (n),Positivo (% del grupo),Negativo (n),Negativo (% del grupo),Faltante (n),Faltante (% del grupo),Mediana (no faltantes)
0,PASAJEROS = 0,PAX_LIB,11639,11505,98.8487,0,0.0000,0,0.0,134,1.1513,0.00
1,PASAJEROS = 0,CAR_LIB,11639,11639,100.0000,0,0.0000,0,0.0,0,0.0000,0.00
2,PASAJEROS = 0,CARGA (Ton),11639,8138,69.9201,3501,30.0799,0,0.0,0,0.0000,0.00
3,PASAJEROS = 0,CORREO,11639,11505,98.8487,0,0.0000,0,0.0,134,1.1513,0.00
4,PASAJEROS > 0,PAX_LIB,54617,37819,69.2440,0,0.0000,0,0.0,16798,30.7560,0.00
5,PASAJEROS > 0,CAR_LIB,54617,54617,100.0000,0,0.0000,0,0.0,0,0.0000,0.00
6,PASAJEROS > 0,CARGA (Ton),54617,22978,42.0712,31639,57.9288,0,0.0,0,0.0000,0.19
7,PASAJEROS > 0,CORREO,54617,37819,69.2440,0,0.0000,0,0.0,16798,30.7560,0.00


Los resultados muestran 49.202 filas (19,59 %) con PASAJEROS igual a cero y 201.915 (80,41 %) con pasajeros positivos; no hay negativos ni faltantes en esta variable. La tasa de cero es 28,86 % en registros internacionales y 1,71 % en nacionales; cerca del 97,01 % de las filas con pasajeros cero son internacionales. Por año, las tasas más altas corresponden a 2020 (33,61 %) y 2021 (28,71 %). Al agregar todos los años, la proporción mensual varía entre 18,26 % en enero y 20,48 % en junio. Son asociaciones descriptivas; estos datos no identifican sus causas.

LATAM CARGO presenta 6.548 de 6.548 registros con PASAJEROS igual a cero. Entre las rutas, MIA–SCL tiene 2.925 ceros entre 5.297 registros (55,22 %). Estos patrones preliminares pueden orientar preguntas posteriores, pero el nombre del operador, la ruta o un valor cero no bastan para clasificar el tipo de operación.

En el grupo con pasajeros cero, CARGA (Ton) es positiva en 43,09 % de las filas, CORREO en 4,17 % y CAR_LIB en 1,88 %; PAX_LIB es positivo en 0,09 %. La observación confirma que carga o correo pueden tener valores positivos cuando PASAJEROS es cero, pero no permite afirmar que sean vuelos de carga. El valor negativo de CARGA (Ton) también pertenece a este grupo. PAX_LIB y CORREO tienen 977 faltantes entre las filas con pasajeros cero y 56.085 entre las filas con pasajeros positivos; su cantidad no explica por sí sola por qué faltan.

Estas diferencias pueden orientar análisis posteriores, pero no justifican eliminar variables ni registros.

### Posible filtrado de registros con pocos pasajeros

No aplicaremos un umbral mínimo ni excluiremos los registros con pocos pasajeros o con cero. Filtrarlos podría facilitar el estudio de rutas de mayor volumen, pero también ocultar rutas de baja demanda, variación estacional u operaciones alternativas. Sin conocer el significado de los ceros y valores bajos, excluirlos podría sesgar las comparaciones. La decisión queda pendiente para después de estudiar la distribución de PASAJEROS en la sección 2.

### Cobertura temporal y geográfica

Verificaremos los años y meses presentes y el alcance geográfico usando las variables de origen, destino y país disponibles. Esto permite confirmar si el periodo descrito en el README coincide con el archivo y si hay cobertura útil para comparar rutas nacionales e internacionales.

In [ ]:
if "Año" in df.columns:
    print(f"Años observados: {df['Año'].min()}–{df['Año'].max()}")
    print(f"Cantidad de años distintos: {df['Año'].nunique()}")
if "Mes" in df.columns:
    print("Meses observados:", sorted(df["Mes"].dropna().unique().tolist()))

for columna in ["NAC", "ORIG_1", "DEST_1", "ORIG_1_PAIS", "DEST_1_PAIS"]:
    if columna in df.columns:
        print(f"{columna}: {df[columna].nunique(dropna=True)} categorías distintas")

if {"ORIG_1", "DEST_1"}.issubset(df.columns):
    rutas_distintas = df[["ORIG_1", "DEST_1"]].drop_duplicates().shape[0]
    print(f"Pares origen-destino distintos: {rutas_distintas:,}")

Años observados: 1984–1994
Cantidad de años distintos: 11
Meses observados: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
NAC: 2 categorías distintas
ORIG_1: 243 categorías distintas
DEST_1: 246 categorías distintas
ORIG_1_PAIS: 64 categorías distintas
DEST_1_PAIS: 64 categorías distintas
Pares origen-destino distintos: 870


### Combinaciones de año y mes sin registros

Que aparezcan los 12 meses en el dataset no garantiza que cada año tenga observaciones en todos ellos. Compararemos todas las combinaciones año-mes posibles entre el primer y el último año observados con las combinaciones presentes en los datos.

In [ ]:
periodo_anios = range(int(df["Año"].min()), int(df["Año"].max()) + 1)
combinaciones_esperadas = pd.MultiIndex.from_product([periodo_anios, range(1, 13)], names=["Año", "Mes"])
combinaciones_observadas = pd.MultiIndex.from_frame(df[["Año", "Mes"]].drop_duplicates())
meses_ausentes = combinaciones_esperadas.difference(combinaciones_observadas)
print(f"Combinaciones año-mes sin registros: {len(meses_ausentes):,}")
if len(meses_ausentes):
    display(meses_ausentes.to_frame(index=False))
else:
    print("No se encontraron meses ausentes dentro del periodo observado.")

Combinaciones año-mes sin registros: 1


,Año,Mes
0,1994,12


La cobertura agregada de años y meses se verifica con esta tabla de combinaciones. Si existen meses ausentes, podrían interrumpir series temporales y afectar comparaciones mensuales; su ausencia también puede responder a la disponibilidad histórica del registro y debe considerarse al interpretar tendencias. Si no aparecen combinaciones ausentes, eso confirma continuidad mensual en el periodo evaluado, aunque no garantiza que cada ruta u operador tenga datos todos los meses.

### Síntesis de estructura y calidad

El dataset contiene **251.117 registros y 25 variables** (4 enteras, 4 decimales y 17 de texto según pandas), con cobertura entre **1984 y 2025** y sin combinaciones año-mes ausentes. PASAJEROS aparece en todas las filas: **49.202 ceros (19,59 %) y 201.915 valores positivos (80,41 %)**, sin valores negativos ni faltantes.

**Calidad confirmada:** hay **18 filas duplicadas exactas (0,01 %)**; espacios exteriores en **128 valores de Operador**, **2 de ORIG_1_N** y **2 de DEST_2_N**; **57.062 faltantes (22,72 %)** en cada una de PAX_LIB y CORREO; un valor negativo de CARGA (Ton), leído por pandas como **−1.726**; y valores extremos de CORREO de **4.148.245.086** y **2.457.561.050**. El criterio IQR identifica **35.514 valores de PASAJEROS por sobre el umbral superior (14,14 %)**: son extremos estadísticos potenciales, no errores confirmados.

**Ceros y contexto:** PAX_LIB tiene 158.925 ceros, 35.130 positivos y 57.062 faltantes; CAR_LIB, 246.589 ceros y 4.528 positivos; CARGA (Ton), 117.293 ceros, 133.823 positivos y 1 negativo; CORREO, 182.280 ceros, 11.775 positivos y 57.062 faltantes; Distancia es positiva en las 251.117 filas. La tasa de cero de PASAJEROS es mayor en registros internacionales (28,86 %) que nacionales (1,71 %), y es más alta en 2020 (33,61 %) y 2021 (28,71 %). Algunos operadores y rutas concentran registros con pasajeros cero. En ese grupo también hay carga positiva en 43,09 % de las filas y correo positivo en 4,17 %. Son patrones observados, no evidencia suficiente para asignar un tipo de operación o explicar causalidad.

**Decisiones pendientes:** investigar en la fuente el valor negativo de carga, los extremos de correo, los duplicados y el significado de los registros con cero pasajeros. Los espacios exteriores podrían corregirse como formato después de justificar y aprobar una transformación; no se normalizaron aquí. PAX_LIB, CAR_LIB, CARGA (Ton), CORREO y Distancia podrían aportar contexto, pero su inclusión en los análisis principales debe evaluarse frente a la pregunta, sus faltantes y su proporción de ceros. No se eliminó ninguna columna ni fila, no se imputaron faltantes y no se filtraron registros por pasajeros. El tratamiento de extremos y cualquier filtro deben esperar al análisis univariado de la sección 2.

## Limpieza y preparación del dataset

La limpieza se realizará sobre una copia independiente del dataset, conservando intactos el archivo original y el DataFrame usado en la exploración. Se recortarán espacios externos en columnas de texto, se excluirán PAX_LIB, CAR_LIB y CORREO por no ser centrales para la pregunta principal y se eliminarán, en pasos separados, las filas con nulos restantes y los duplicados exactos.

Los seis registros de AEROVIAS DAP asociados a AWC tienen información incompleta en nombres o países geográficos. Como no se pudo verificar qué representa AWC ni una corrección fiable, no se reemplazará el código: esos registros se contabilizarán y excluirán por los valores nulos que permanecen tras retirar las tres columnas. Los duplicados se identificarán únicamente cuando todas las columnas conservadas coincidan después del recorte de espacios.

In [ ]:
from pathlib import Path

ruta_original_limpieza = Path("trafico_historico.csv")
ruta_salida_limpieza = Path("trafico_limpio.csv")

# 1. Cargar el archivo original y registrar sus dimensiones.
df_original_limpieza = pd.read_csv(ruta_original_limpieza, sep=";")
filas_originales_limpieza, columnas_originales_limpieza = df_original_limpieza.shape
print(f"Dimensiones originales: {filas_originales_limpieza:,} filas × {columnas_originales_limpieza} columnas")

# 2. Trabajar sobre una copia; el DataFrame df de la exploración no se modifica.
df_limpio = df_original_limpieza.copy()

# 3. Recortar espacios externos de campos de texto.
columnas_texto_limpieza = df_limpio.select_dtypes(include=["object", "string"]).columns
for columna in columnas_texto_limpieza:
    df_limpio[columna] = df_limpio[columna].str.strip()

# 4. Excluir variables que no son centrales para el análisis principal.
columnas_excluir_limpieza = ["PAX_LIB", "CAR_LIB", "CORREO"]
columnas_eliminadas_limpieza = [
    columna for columna in columnas_excluir_limpieza
    if columna in df_limpio.columns
]
df_limpio = df_limpio.drop(columns=columnas_eliminadas_limpieza)

# 5. Contar y eliminar filas con nulos en las columnas que se conservarán.
mascara_nulos_limpieza = df_limpio.isna().any(axis=1)
filas_nulas_eliminadas_limpieza = int(mascara_nulos_limpieza.sum())
df_limpio = df_limpio.loc[~mascara_nulos_limpieza].copy()

# 6. Eliminar únicamente filas exactamente duplicadas entre las columnas restantes.
filas_antes_duplicados_limpieza = len(df_limpio)
df_limpio = df_limpio.drop_duplicates().copy()
duplicados_eliminados_limpieza = filas_antes_duplicados_limpieza - len(df_limpio)

# 7. Guardar y recargar el archivo para validar el resultado escrito.
df_limpio.to_csv(ruta_salida_limpieza, sep=";", index=False, encoding="utf-8")
df_limpio_verificacion = pd.read_csv(ruta_salida_limpieza, sep=";")

nulos_restantes_limpieza = int(df_limpio_verificacion.isna().sum().sum())
duplicados_restantes_limpieza = int(df_limpio_verificacion.duplicated().sum())
columnas_excluidas_restantes_limpieza = [
    columna for columna in columnas_excluir_limpieza
    if columna in df_limpio_verificacion.columns
]
validacion_limpieza_correcta = (
    nulos_restantes_limpieza == 0
    and duplicados_restantes_limpieza == 0
    and len(columnas_excluidas_restantes_limpieza) == 0
    and df_limpio.shape == df_limpio_verificacion.shape
)

resumen_limpieza = pd.Series({
    "Filas originales": filas_originales_limpieza,
    "Columnas originales": columnas_originales_limpieza,
    "Columnas eliminadas": len(columnas_eliminadas_limpieza),
    "Filas eliminadas por nulos": filas_nulas_eliminadas_limpieza,
    "Filas eliminadas por duplicados exactos": duplicados_eliminados_limpieza,
    "Filas finales": df_limpio_verificacion.shape[0],
    "Columnas finales": df_limpio_verificacion.shape[1],
    "Valores nulos restantes": nulos_restantes_limpieza,
    "Duplicados exactos restantes": duplicados_restantes_limpieza
}, name="Resultado")
display(resumen_limpieza.to_frame())

print("Columnas excluidas presentes en el archivo final:", columnas_excluidas_restantes_limpieza)

if validacion_limpieza_correcta:
    print("VALIDACIÓN CORRECTA: el archivo recargado no contiene nulos, duplicados exactos ni las columnas excluidas.")
else:
    print("AVISO: falló al menos una validación; revisar los resultados antes de usar el archivo.")
    print("Nulos restantes:", nulos_restantes_limpieza)
    print("Duplicados exactos restantes:", duplicados_restantes_limpieza)
    print("Columnas excluidas presentes:", columnas_excluidas_restantes_limpieza)
    print("Dimensiones coinciden:", df_limpio.shape == df_limpio_verificacion.shape)

Dimensiones originales: 105,967 filas × 25 columnas


,Resultado
Filas originales,105967
Columnas originales,25
Columnas eliminadas,3
Filas eliminadas por nulos,1
Filas eliminadas por duplicados exactos,4
Filas finales,105962
Columnas finales,22
Valores nulos restantes,0
Duplicados exactos restantes,0


Columnas excluidas presentes en el archivo final: []
VALIDACIÓN CORRECTA: el archivo recargado no contiene nulos, duplicados exactos ni las columnas excluidas.


La ejecución conserva el CSV original y genera un archivo independiente. Se excluyeron **3 columnas**, se eliminaron **6 filas por valores nulos** y, después, **18 filas por duplicados exactos**. El archivo recargado tiene **251.093 filas y 22 columnas**; las validaciones confirman **0 nulos, 0 duplicados exactos y ausencia de PAX_LIB, CAR_LIB y CORREO**.

Los seis registros retirados por nulos corresponden a AEROVIAS DAP e incluyen AWC en los campos de origen o destino, con nombres o países incompletos. No se imputó información ni se sustituyó AWC. Los duplicados se contaron después de retirar los nulos y no se eliminaron filas por similitud o por compartir ruta, mes u operador. El recorte se limitó a espacios externos, incluidos espacios Unicode, sin alterar códigos, mayúsculas ni nombres. No se aplicaron filtros adicionales.

#2. Análisis univariado

#3. Relaciones entre variables

#4. Componente temporal o contextual
